# University of Waterloo History Workshop

Notebook v.1.0 (2026-11)

Jupyter notebook by: R. Antonio Muñoz Gómez. (Cataloguing and Metadata Librarian. University of Waterloo)

Workshop delivered by: Mike Chee (Research and Learning Librarian. University of Waterloo) and R. Antonio Muñoz Gómez. (Cataloguing and Metadata Librarian. University of Waterloo)

## Introduction

This notebook was created for a graduate-student workshop on working with sources in languages other than English using computer-generated translation.

During the workshop, different approaches are discussed, including the use of AI. The objective is to present students with a variety of approaches, and showcase the pros and cons afforded by each of these digital methods.

Further, the notebook includes a section on Preliminary Analysis, providing examples of how, once data exists in a machine-readable format, it can be analysed and manipulated in different ways to obtain insights from it.

Students are encouraged to build on this knowledge, and make a copy of the notebook that they can modify as they test new code and methods that better suit their individual needs.

For example, the University of Waterloo's Centre for Education in Mathematics and Computing has an excellent [Python from scratch](https://open.cs.uwaterloo.ca/python-from-scratch/) course.

## Notebook structure

The notebook has four main sections:

- Part 1. File preparation and text extraction : This code will set up the source file and file-naming conventions for outputs, open the .pdf file, verify whether a text layer exists and, if applicable, extract it to a .txt file for translation.
- Part 2. Text cleanup : This code performs some basic cleanup tasks to assist the translation process.
- Part 3. Translation : This code takes the cleaned up text and translates it into English (other language pairings are available).
- Part 4. Basic analysis : This code performs some simple analysis tasks such as: word frequencies, word cloud, and named-entity identification.

The Jupyter notebook provides a structured and reproducible framework that students can use for their projects, as well as allowing for flexibility to have code modified and tailored to specific student/project needs.

## Before we get started

You will need to add the test file (example: Louis-Riel-martyr.pdf) to the notebook.

### If you are working in Google Colab:

1. From the left-hand menu options, choose Files (folder icon)
2. Right click on the panel and choose the option to Upload
3. Select the .pdf file that we will work with
4. You may also upload an additional .pdf file if you want to do more testing

## Part 1 : File preparation and text extraction

In this workshop, we are working with .pdf files. As discussed during the workshop, different possibilities exist in the making of this kind of file, including:

- A born-digital file with an attached text layer that can be extracted
- A digitized file with a text layer generated through an OCR process
- A file with human-readable text, but saved as an image without a text layer that can be meaningfully extracted without doing OCR first

### Step 1 : Installing and importing libraries

In programming, 'libraries' refer to code that someone else has written and made available for re-use. Rather than having to write all that code from scratch, you can use it by 'importing' the libraries into your own code.

To successfully import a library, first you must make sure that it has been installed.

The code below will try to import the following library (follow the link if you want to learn more about this library and what it is used for):

* [PYPDF](https://pypi.org/project/pypdf/)

If the library cannot be found in the system, the code will first install and then import it.

In [ ]:
try:
    from pypdf import PdfReader
except ImportError:
    %pip install pypdf
    from pypdf import PdfReader

### Step 2 : Indicate the name of the file that you want to extract text from

In the code below, substitute `[YOUR-FILENAME]` with the filename of the .pdf that you'll be using. Make sure to keep the `.pdf` file extension.

In [ ]:
# Set the name of the source PDF file
sourceFileName = 'Louis-Riel-martyr.pdf'

# Open the source PDF file
sourceFileReader = PdfReader(sourceFileName)

### Step 3 : Choose output file-naming prefix method

When you work with multiple files, having a consistent file-naming structure can help you easily find and identify them as you continue your research.

In this workshop, you start with a .pdf file which will generate different outputs including:

- An extracted text layer
- A cleaned up version of the text layer
- A translation
- Different files with preliminary analysis

In this workshop, we are also working with a single page and the full text for the above tasks, so that we can easily observe the outputs with the single page, and accommodate the potentially longer time to complete these tasks on full-text depending on the length of the file.

In the following code of block, you can choose between an automatic file-naming convention (the code will take your original PDF file name as the basis for naming outputs), or you can choose to manually enter your own prefix.

By default, the code is set to 'automatic'. You may change the code as in the following example:

```
outputFilePrefix = 'your-custom-text'
```

In [ ]:
# Set the prefix for output files
outputFilePrefix = 'automatic'

#### Step 3.1 : Prepare output prefix

The following code will build the filename prefix based on your indicated preference.

In [ ]:
from pathlib import Path
import re
import unicodedata

if outputFilePrefix == 'automatic':

    # Remove the file extension
    fileName = Path(sourceFileName).stem

    # Normalize accented characters
    fileName = unicodedata.normalize('NFKD', fileName)
    fileName = fileName.encode('ascii', 'ignore').decode('ascii')

    # Divide the filename into alphabetic and numeric parts
    fileParts = re.findall(r'[A-Za-z0-9]+', fileName)

    # Use the first two parts as the output prefix
    outputFilePrefix = '-'.join(fileParts[:2]).lower()

else:

    # Normalize a manually entered prefix
    outputFilePrefix = unicodedata.normalize(
        'NFKD',
        outputFilePrefix
    )

    outputFilePrefix = outputFilePrefix.encode(
        'ascii',
        'ignore'
    ).decode('ascii')

    # Convert spaces and punctuation to consistent kebab case
    fileParts = re.findall(
        r'[A-Za-z0-9]+',
        outputFilePrefix
    )

    outputFilePrefix = '-'.join(fileParts).lower()

# Confirm the output prefix
print(f'Source file: {sourceFileName}')
print(f'Output file prefix: {outputFilePrefix}')


### Step 4 : Testing the file

The following blocks of code help us test that the file is suitable for extracting a text layer that we can use for translation and analysis.

#### Step 4.1 : Counting the number of pages

In [ ]:
pageCount = len(sourceFileReader.pages)
print('The document contains', pageCount, 'pages.')

#### Step 4.2 : Reading a single page

We have confirmed that Python can open the PDF and have counted the number of pages. We will now test whether we can extract computer-readable text from one of those pages.

The code below selects one page from the PDF, extracts any text available from that page, and prints the result.

Look at the output. Is the text readable? Are there errors? Does it appear to correspond to the text visible on the PDF page?

Note: This code is not performing OCR. It is extracting a text layer that already exists in the PDF.

**Before running the code:**

- Read it and try to guess which page we will extract from the document.
- How can you tell?

Now, run the code.

In [ ]:
pageObj=sourceFileReader.pages[5]
pageText=pageObj.extract_text()

print(pageText)

#### Step 4.3 : Reading the full text

In [ ]:
fullText= ""

for i in range(pageCount):
    page=sourceFileReader.pages[i]
    pageTextExtracted=page.extract_text() or ""
    fullText+= pageTextExtracted + "\n\n"

print(fullText)

### Step 5 : Saving the text

The following code will save the single page and the full text extracted above.

In [ ]:
# Save the single-page extraction
with open(f'{outputFilePrefix}-extracted-page.txt', 'w', encoding='utf-8') as f:
    f.write(pageText)

# Save the full-text extraction
with open(f'{outputFilePrefix}-extracted-full.txt', 'w', encoding='utf-8') as f:
    f.write(fullText)

print('The extracted text files have been saved.')

## Part 2 : Text cleanup

Text extracted from a PDF file may have different quirks inherited from the layout of the original document, such as line breaks within paragraphs or extra spaces.

Before translating the text, we can perform some basic cleanup. The following code will:

- preserve paragraph breaks
- join lines that belong to the same paragraph
- remove unnecessary spaces

Rather than changing the original extracted text, we will create new files that can be used for comparison purposes.

Text cleanup should be approached cautiously. Features that appear to be formatting problems may sometimes reflect meaningful characteristics of the source. This is particularly important when working with historical documents.

The tasks performed in this block of code are a bare-minimum attempt to have reasonably-useful text to translate from. A full cleanup process may involve several stages of looking at the outputs, identifying patterns of 'noise' and manually or programmatically fixing them. Often, such 'noise' patterns will be specific to the file and/or to the source of the file (example: .pdf files from the same journal are likely to have similar formatting and structure and it might be somewhat easier to anticipate common cleanup tasks required for that group of files).

### Step 1 : Single page

#### Step 1.1 : Single-page cleanup

In [ ]:
# Split the page text into paragraphs
paragraphs = pageText.split('\n\n')

# Join line breaks within each paragraph and remove extra whitespace
cleanedParagraphs = [
    ' '.join(paragraph.split())
    for paragraph in paragraphs
]

# Reassemble the paragraphs
cleanedPageText = '\n\n'.join(cleanedParagraphs)

#### Step 1.2 : Single-page cleanup inspection

In [ ]:
print(cleanedPageText)

#### Step 1.3 : Single-page cleanup saving

In [ ]:
# Save the cleaned text from the selected page
with open(f'{outputFilePrefix}-cleaned-page.txt', 'w', encoding='utf-8') as outputFile:
    outputFile.write(cleanedPageText)

print('Cleaned page text has been saved.')

### Step 2 : Full text

#### Step 2.1 : Full-text cleanup

In [ ]:
# Split the full text into paragraphs
paragraphs = fullText.split('\n\n')

# Join line breaks within each paragraph and remove extra whitespace
cleanedParagraphs = [
    ' '.join(paragraph.split())
    for paragraph in paragraphs
]

# Reassemble the paragraphs
cleanedFullText = '\n\n'.join(cleanedParagraphs)

#### Step 2.2 : Full-text cleanup inspection

In [ ]:
print(cleanedFullText[:10000])

#### Step 2.3 : Full-text cleanup saving

In [ ]:
# Save the cleaned text from the full document
with open(f'{outputFilePrefix}-cleaned-full.txt', 'w', encoding='utf-8') as outputFile:
    outputFile.write(cleanedFullText)

print('Cleaned full text has been saved.')

## Part 3 : Translation

### Step 1 : Installing and importing libraries

Similar to the libraries that we imported for extracting text, this code will check whether a library called Argos Translate is available in the current Python environment. If the library has already been installed, it will simply be imported. If it is missing, the notebook will install it before importing the components needed for translation.

Argos Translate uses separate language-model packages. This step installs the translation software itself; we will check for and install the required language model later, after selecting the source and target languages.

For more details about this library, you can check the following link:

- [Argos Translate](https://github.com/argosopentech/argos-translate)

In [ ]:
try:
    import argostranslate.package
    import argostranslate.translate
    print('Argos Translate is already installed and ready to use.')

except ImportError:
    print('Argos Translate is not installed. Installing it now...')

    %pip install argostranslate

    import argostranslate.package
    import argostranslate.translate
    print('Argos Translate has been installed and is ready to use.')

### Step 2 : Examine available translation packages

Argos Translate uses language-model packages for particular source and target languages. Before selecting the languages for translation, we can retrieve the available packages and display the language names and codes associated with each translation pair.

The language code in parentheses will be used in the next step to specify the source and target languages.

In [ ]:
# Update the Argos Translate package index
# and retrieve the available translation packages
try:
    argostranslate.package.update_package_index()
    availablePackages = argostranslate.package.get_available_packages()

    # Create and sort a list of available translation pairs
    routes = sorted({
        f'{package.from_name} ({package.from_code}) -> '
        f'{package.to_name} ({package.to_code})'
        for package in availablePackages
    })

    print('Available translation language pairs in Argos Translate:\n')

    for route in routes:
        print(route)

except Exception as error:
    print(f'Could not retrieve the available translation packages: {error}')

### Step 3 : Set the source (`from_code`) and target (`to_code`) languages

Using the language codes shown in the previous step, specify the language of the source text and the language into which it will be translated.

In this example, the source text is in French (fr) and will be translated into English (en). Change these codes as needed for the text you are working with.

In [ ]:
# Set the source and target languages
from_code = 'fr'
to_code = 'en'

### Step 4 : Check and install the translation model

Argos Translate requires a language-model package for the selected source and target languages. This step checks whether the required translation model is already installed.

If the model is installed, the notebook will continue without reinstalling it. If it is missing, the notebook will find the appropriate package in the package index, download it, and install it.

Downloading a new model requires an internet connection. Once the model has been installed, it can be used for offline translation.

In [ ]:
# Check whether the selected translation model is already installed
installedPackages = argostranslate.package.get_installed_packages()

modelInstalled = any(
    package.from_code == from_code and package.to_code == to_code
    for package in installedPackages
)

if modelInstalled:
    print(
        f'The translation model from {from_code} to {to_code} '
        'is already installed.'
    )

else:
    print(
        f'The translation model from {from_code} to {to_code} '
        'is not installed. Installing it now...'
    )

    # Find the selected translation package
    selectedPackage = next(
        (
            package for package in availablePackages
            if package.from_code == from_code
            and package.to_code == to_code
        ),
        None
    )

    if selectedPackage is not None:
        downloadPath = selectedPackage.download()
        argostranslate.package.install_from_path(downloadPath)

        print(
            f'The translation model from {from_code} to {to_code} '
            'has been installed.'
        )

    else:
        print(
            f'No direct translation model from {from_code} to {to_code} '
            'was found.'
        )

### Step 5 : Single page

#### Step 5.1 : Single-page translation

We will first test the translation process using the cleaned text from a single page. The cleaned text created earlier is translated using the source and target languages selected above.

Testing the process on a single page allows us to examine the results before applying the same process to the full document.

In [ ]:
import logging

# Suppress informational messages from the translation libraries
logging.getLogger('argostranslate').setLevel(logging.ERROR)
logging.getLogger('stanza').setLevel(logging.ERROR)

print('Translating cleaned page...')

# Translate the cleaned page
translatedPageText = argostranslate.translate.translate(
    cleanedPageText,
    from_code,
    to_code
)

print('Translation complete.')

#### Step 5.2 : Single-page translation inspection

In [ ]:
print(translatedPageText)

#### Step 5.3 : Single-page translation saving

In [ ]:
# Save the translated text from the selected page
with open(f'{outputFilePrefix}-translated-page.txt', 'w', encoding='utf-8') as outputFile:
    outputFile.write(translatedPageText)

print('Translated page text has been saved.')

### Step 6 : Full text

#### Step 6.1 : Full-text translation

After successfully testing the translation process on a single page, we can apply the same process to the cleaned text from the full document.

Depending on the length of the document and the computer being used, translating the full text may take considerably longer than translating a single page. A message will indicate when the translation begins and when it has been completed.

In [ ]:
import threading
import time

print(
    'Translating cleaned full text. This operation may take several minutes depending on the document length.'
)
print(
    'A progress update will be displayed once per minute until the translation is complete.'
)

# Record when the translation starts
startTime = time.time()

# Create a signal used to stop the timer
stopTimer = threading.Event()

# Display an elapsed-time message every minute
def showElapsedTime():
    while not stopTimer.wait(60):
        elapsedMinutes = int((time.time() - startTime) // 60)
        print(
            f'{elapsedMinutes} minute(s) elapsed. '
            'Translation is still in progress...'
        )

# Run the timer in the background
timerThread = threading.Thread(
    target=showElapsedTime,
    daemon=True
)
timerThread.start()

# Temporarily suppress logging messages
logging.disable(logging.CRITICAL)

try:
    translatedFullText = argostranslate.translate.translate(
        cleanedFullText,
        from_code,
        to_code
    )

except KeyboardInterrupt:
    print('\nTranslation interrupted by the user.')
    raise

except Exception as error:
    print(f'\nTranslation failed: {error}')
    raise

else:
    elapsedSeconds = time.time() - startTime
    elapsedMinutes = elapsedSeconds / 60

    print(
        f'\nTranslation complete. '
        f'Total time: {elapsedMinutes:.1f} minute(s).'
    )

finally:
    # Stop the timer and restore logging
    stopTimer.set()
    logging.disable(logging.NOTSET)

#### Step 6.2 : Full-text translation inspection

In [ ]:
print(translatedFullText[0:25000])

#### Step 6.3 : Full-text translation saving

We can save the single-page and full-document translations as separate UTF-8 encoded text files. Keeping these files alongside the extracted and cleaned versions provides evidence of each transformation applied to the source text.

Machine-generated translations should be reviewed carefully before they are used as research evidence or shared as finished translations.

In [ ]:
# Save the translated text from the full document
with open(f'{outputFilePrefix}-translated-full.txt', 'w', encoding='utf-8') as outputFile:
    outputFile.write(translatedFullText)

print('Translated full text has been saved.')

## Part 4 : Preliminary text analysis

Once a document has been converted into machine-readable text, computational methods can help us identify patterns within it. The following examples use word frequencies and named entities to examine both the original-language text and its machine translation.

Comparing the results may reveal differences introduced by translation or by the language-specific analysis tools. These outputs should be treated as exploratory evidence rather than definitive interpretations of the source.

The following blocks of code produce some common analyses used in Digital Humanities and are just meant to exemplify some of the possibilities afforded by machine-readable text availability.

### Step 1 : Word frequencies and word clouds

In [ ]:
import sys
import subprocess
import importlib.util
from collections import Counter

# Install any missing analysis libraries
requiredLibraries = {
    'spacy': 'spacy',
    'pandas': 'pandas',
    'matplotlib': 'matplotlib',
    'wordcloud': 'wordcloud'
}

for moduleName, packageName in requiredLibraries.items():
    if importlib.util.find_spec(moduleName) is None:
        print(f'Installing {packageName}...')
        subprocess.check_call([
            sys.executable,
            '-m',
            'pip',
            'install',
            packageName
        ])

# Import the analysis libraries
import spacy
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# Associate language codes with spaCy language models
languageModels = {
    'de': 'de_core_news_sm',
    'en': 'en_core_web_sm',
    'es': 'es_core_news_sm',
    'fr': 'fr_core_news_sm',
    'it': 'it_core_news_sm',
    'nl': 'nl_core_news_sm',
    'pt': 'pt_core_news_sm'
}

# Check whether models are available for both languages
if from_code not in languageModels:
    raise ValueError(
        f'No spaCy model has been configured for the source language: '
        f'{from_code}'
    )

if to_code not in languageModels:
    raise ValueError(
        f'No spaCy model has been configured for the target language: '
        f'{to_code}'
    )

originalModelName = languageModels[from_code]
translatedModelName = languageModels[to_code]

# Install and load a language model if necessary
def loadLanguageModel(modelName):
    try:
        return spacy.load(modelName)

    except OSError:
        print(f'Installing the language model {modelName}...')

        from spacy.cli import download
        download(modelName)

        return spacy.load(modelName)

# Load the source- and target-language models
originalNLP = loadLanguageModel(originalModelName)
translatedNLP = loadLanguageModel(translatedModelName)

# Allow spaCy to process texts longer than its default limit
originalNLP.max_length = max(
    originalNLP.max_length,
    len(cleanedFullText) + 1000
)

translatedNLP.max_length = max(
    translatedNLP.max_length,
    len(translatedFullText) + 1000
)

print('Analyzing the original and translated texts...')

# Process both versions of the document
originalDoc = originalNLP(cleanedFullText)
translatedDoc = translatedNLP(translatedFullText)

# Extract meaningful words and exclude punctuation, numbers,
# spaces, and common stop words
originalWords = [
    token.lemma_.lower()
    for token in originalDoc
    if token.is_alpha
    and not token.is_stop
    and token.lemma_.strip()
]

translatedWords = [
    token.lemma_.lower()
    for token in translatedDoc
    if token.is_alpha
    and not token.is_stop
    and token.lemma_.strip()
]

# Count the words in each document
originalWordCounts = Counter(originalWords)
translatedWordCounts = Counter(translatedWords)

# Create tables containing the 25 most frequent words
originalWordFrequencyTable = pd.DataFrame(
    originalWordCounts.most_common(25),
    columns=['Word', 'Count']
)

translatedWordFrequencyTable = pd.DataFrame(
    translatedWordCounts.most_common(25),
    columns=['Word', 'Count']
)

print('Most frequent words in the original-language text:')
display(originalWordFrequencyTable)

print('Most frequent words in the translated text:')
display(translatedWordFrequencyTable)

# Create word clouds from the complete frequency results
originalWordCloud = WordCloud(
    width=1200,
    height=600,
    background_color='white'
).generate_from_frequencies(originalWordCounts)

translatedWordCloud = WordCloud(
    width=1200,
    height=600,
    background_color='white'
).generate_from_frequencies(translatedWordCounts)

# Display the two word clouds side by side
wordCloudComparison, axes = plt.subplots(
    1,
    2,
    figsize=(16, 7)
)

axes[0].imshow(
    originalWordCloud,
    interpolation='bilinear'
)
axes[0].set_title(
    f'Original-language text ({from_code})',
    fontsize=14
)
axes[0].axis('off')

axes[1].imshow(
    translatedWordCloud,
    interpolation='bilinear'
)
axes[1].set_title(
    f'Translated text ({to_code})',
    fontsize=14
)
axes[1].axis('off')

plt.tight_layout()
plt.show()

print('Word-frequency analysis complete.')

### Step 2 : Named entities

Named-entity recognition attempts to identify people, organizations, and places mentioned in a text. The results are generated automatically and may contain omissions, incorrect identifications, or incorrect classifications.

In [ ]:
# Translate spaCy entity labels into broader categories
entityLabels = {
    'PER': 'Person',
    'PERSON': 'Person',
    'ORG': 'Organization',
    'LOC': 'Place',
    'GPE': 'Place',
    'FAC': 'Place'
}

# Extract and count selected named entities
def createEntityTable(document):
    entities = []

    for entity in document.ents:
        entityType = entityLabels.get(entity.label_)

        # Include only people, organizations, and places
        if entityType is not None:
            entityName = ' '.join(entity.text.split())

            if entityName:
                entities.append(
                    (entityName, entityType)
                )

    entityCounts = Counter(entities)

    entityTable = pd.DataFrame(
        [
            {
                'Entity': entityName,
                'Type': entityType,
                'Count': count
            }
            for (entityName, entityType), count
            in entityCounts.items()
        ],
        columns=['Entity', 'Type', 'Count']
    )

    if not entityTable.empty:
        entityTable = entityTable.sort_values(
            by=['Count', 'Entity'],
            ascending=[False, True]
        ).reset_index(drop=True)

    return entityTable

# Create entity tables for both versions of the text
originalEntityTable = createEntityTable(originalDoc)
translatedEntityTable = createEntityTable(translatedDoc)

print('Named entities in the original-language text:')
display(originalEntityTable)

print('Named entities in the translated text:')
display(translatedEntityTable)

print('Named-entity analysis complete.')

### Step 3 : Save files resulting from preliminary analysis

Save the tables and word clouds in a separate folder for comparison and further examination.

In [ ]:
from pathlib import Path

# Create a document-specific folder for the analysis outputs
analysisFolder = Path('PreliminaryAnalysis') / outputFilePrefix
analysisFolder.mkdir(parents=True, exist_ok=True)

# Save the word-frequency tables
originalWordFrequencyTable.to_csv(
    analysisFolder / 'original-word-frequencies.csv',
    index=False,
    encoding='utf-8'
)

translatedWordFrequencyTable.to_csv(
    analysisFolder / 'translated-word-frequencies.csv',
    index=False,
    encoding='utf-8'
)

# Save the named-entity tables
originalEntityTable.to_csv(
    analysisFolder / 'original-named-entities.csv',
    index=False,
    encoding='utf-8'
)

translatedEntityTable.to_csv(
    analysisFolder / 'translated-named-entities.csv',
    index=False,
    encoding='utf-8'
)

# Save the individual word clouds
originalWordCloud.to_file(
    analysisFolder / 'original-wordcloud.png'
)

translatedWordCloud.to_file(
    analysisFolder / 'translated-wordcloud.png'
)

# Save a comparison image containing both word clouds
wordCloudComparison.savefig(
    analysisFolder / 'wordcloud-comparison.png',
    dpi=300,
    bbox_inches='tight'
)

print(
    f'Preliminary analysis files saved in '
    f'PreliminaryAnalysis/{outputFilePrefix}.'
)